주요 출력:
- horizon별 독립 XGBoost classifier의 test 성능 확인
- `y_t`, `y_t_plus_1`, `y_t_plus_2` 및 within t+2 기준 permutation feature importance
- horizon별 XGBoost built-in gain importance
- calibration, decision curve analysis(DCA), net reclassification improvement(NRI)
- `RUN_SHAP = True`일 때 Tree SHAP feature importance

In [ ]:
from pathlib import Path
import json
import random
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

warnings.filterwarnings("ignore")


In [ ]:
# 작업 위치
PROJECT_DIR = Path.cwd().resolve().parent
DATA_SPLIT_DIR = PROJECT_DIR / "processed" / "data_split"
CLEAN_DATA_DIR = PROJECT_DIR / "models" / "clean_data"
MODEL_DIR = PROJECT_DIR / "models"
MODELING_OUTPUT_DIR = PROJECT_DIR / "outputs" / "modeling"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "model_interpretation"
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_SPLIT_DIR:", DATA_SPLIT_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# 설정값 (config)
RANDOM_STATE = 42
HORIZONS = ["y_t", "y_t_plus_1", "y_t_plus_2"]
HORIZON_LABELS = {
    "y_t": "t",
    "y_t_plus_1": "t+1",
    "y_t_plus_2": "t+2",
}

MODEL_PATH = MODEL_DIR / "xgb_multi_horizon.joblib"
FEATURE_COLUMNS_PATH = CLEAN_DATA_DIR / "lstm_feature_columns.json"

EXPLAIN_SAMPLE_SIZE = 512
PERMUTATION_REPEATS = 3
TOP_N_PLOT = 25

CALIBRATION_N_BINS = 10
DCA_THRESHOLDS = np.round(np.arange(0.01, 0.81, 0.01), 2)
NRI_RISK_CUTPOINTS = [0.10, 0.20, 0.40]
NRI_REFERENCE_MODEL_NAME = "lstm_gpu"
NRI_REFERENCE_PREDICTION_PATH = MODELING_OUTPUT_DIR / "lstm_gpu_test_predictions.csv"

RUN_SHAP = False
SHAP_EXPLAIN_SIZE = 512
SHAP_HORIZON = "y_t_plus_2"


In [ ]:
# seed 설정
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)


## 전처리 산출물 로딩


In [ ]:
required_files = {
    "X_train": DATA_SPLIT_DIR / "X_train_lstm.npy",
    "X_test": DATA_SPLIT_DIR / "X_test_lstm.npy",
    "y_train_steps": DATA_SPLIT_DIR / "y_train_steps_lstm.npy",
    "y_test_steps": DATA_SPLIT_DIR / "y_test_steps_lstm.npy",
    "y_train_step_mask": DATA_SPLIT_DIR / "y_train_step_mask_lstm.npy",
    "y_test_step_mask": DATA_SPLIT_DIR / "y_test_step_mask_lstm.npy",
    "meta_train": DATA_SPLIT_DIR / "lstm_train_metadata.csv",
    "meta_test": DATA_SPLIT_DIR / "lstm_test_metadata.csv",
}

X_train_sequence = np.load(required_files["X_train"]).astype(np.float32)
X_test_sequence = np.load(required_files["X_test"]).astype(np.float32)
y_train_steps = np.load(required_files["y_train_steps"]).astype(np.float32)
y_test_steps = np.load(required_files["y_test_steps"]).astype(np.float32)
y_train_step_mask = np.load(required_files["y_train_step_mask"]).astype(np.float32)
y_test_step_mask = np.load(required_files["y_test_step_mask"]).astype(np.float32)
meta_train = pd.read_csv(required_files["meta_train"])
meta_test = pd.read_csv(required_files["meta_test"])

# 6_modeling.ipynb의 XGB multi-horizon 모델은 anchor t 시점 feature만 입력으로 사용합니다.
X_train = X_train_sequence[:, -1, :].astype(np.float32)
X_test = X_test_sequence[:, -1, :].astype(np.float32)

with open(FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as f:
    feature_columns = json.load(f)

print("X_train sequence", X_train_sequence.shape)
print("X_test sequence", X_test_sequence.shape)
print("X_train anchor t", X_train.shape)
print("X_test anchor t", X_test.shape)
print("features", len(feature_columns))


In [ ]:
data_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "n_examples": X_train.shape[0],
            "n_features": X_train.shape[1],
            "masked_positive_rate_within_t_plus_2": float(((y_train_steps * y_train_step_mask).max(axis=1) > 0).mean()),
            "nan_count": int(np.isnan(X_train).sum()),
        },
        {
            "split": "test",
            "n_examples": X_test.shape[0],
            "n_features": X_test.shape[1],
            "masked_positive_rate_within_t_plus_2": float(((y_test_steps * y_test_step_mask).max(axis=1) > 0).mean()),
            "nan_count": int(np.isnan(X_test).sum()),
        },
    ]
)
display(data_summary)


## Multi-horizon XGBoost 모델 로딩


In [ ]:
model_payload = joblib.load(MODEL_PATH)
horizon_models = model_payload["models"]

print("loaded:", MODEL_PATH)
print("architecture:", model_payload.get("model_architecture"))
print("input:", model_payload.get("input"))
print("target:", model_payload.get("target"))
print("horizons:", list(horizon_models.keys()))
print("best_params:", model_payload.get("best_params"))


## 예측과 metric helper


In [ ]:
def predict_proba(horizon_models: dict, x: np.ndarray) -> np.ndarray:
    probs = np.zeros((len(x), len(HORIZONS)), dtype=np.float32)
    for idx, horizon in enumerate(HORIZONS):
        probs[:, idx] = horizon_models[horizon].predict_proba(x)[:, 1]
    return probs


def safe_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(int)
    return float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan


def safe_auroc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(int)
    return float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan


def masked_metric_summary(y_steps: np.ndarray, y_prob: np.ndarray, y_mask: np.ndarray) -> dict:
    y_prob_masked = np.where(y_mask.astype(bool), y_prob, 0.0)
    summary = {}
    horizon_auprcs = []
    horizon_aurocs = []
    for idx, horizon in enumerate(HORIZONS):
        active = y_mask[:, idx].astype(bool)
        y_true_h = y_steps[active, idx]
        y_prob_h = y_prob[active, idx]
        auprc = safe_auprc(y_true_h, y_prob_h)
        auroc = safe_auroc(y_true_h, y_prob_h)
        summary[f"{horizon}_auprc"] = auprc
        summary[f"{horizon}_auroc"] = auroc
        summary[f"{horizon}_n"] = int(active.sum())
        horizon_auprcs.append(auprc)
        horizon_aurocs.append(auroc)

    y_true_within = ((y_steps * y_mask).max(axis=1) > 0).astype(int)
    y_prob_within = 1.0 - np.prod(1.0 - y_prob_masked, axis=1)
    summary["macro_auprc"] = float(np.nanmean(horizon_auprcs))
    summary["macro_auroc"] = float(np.nanmean(horizon_aurocs))
    summary["within_t_plus_2_auprc"] = safe_auprc(y_true_within, y_prob_within)
    summary["within_t_plus_2_auroc"] = safe_auroc(y_true_within, y_prob_within)
    return summary


In [ ]:
test_prob = predict_proba(horizon_models, X_test)
test_summary = masked_metric_summary(y_test_steps, test_prob, y_test_step_mask)
display(pd.DataFrame([test_summary]))


## Calibration, DCA, NRI

Test set에서 horizon별 및 within t+2 기준으로 calibration, decision curve analysis, net reclassification improvement를 계산합니다. NRI는 기본적으로 `lstm_gpu_test_predictions.csv`를 reference model로 사용하고, 파일이 없으면 train prevalence reference로 대체합니다.


In [ ]:
def build_prediction_tasks(y_steps: np.ndarray, y_prob: np.ndarray, y_mask: np.ndarray) -> dict:
    tasks = {}
    for idx, horizon in enumerate(HORIZONS):
        active = y_mask[:, idx].astype(bool)
        tasks[horizon] = {
            "label": HORIZON_LABELS[horizon],
            "y_true": y_steps[active, idx].astype(int),
            "y_prob": y_prob[active, idx].astype(float),
            "active": active,
        }

    y_prob_masked = np.where(y_mask.astype(bool), y_prob, 0.0)
    tasks["within_t_plus_2"] = {
        "label": "within t+2",
        "y_true": ((y_steps * y_mask).max(axis=1) > 0).astype(int),
        "y_prob": (1.0 - np.prod(1.0 - y_prob_masked, axis=1)).astype(float),
        "active": np.ones(len(y_steps), dtype=bool),
    }
    return tasks


def calibration_table(tasks: dict, n_bins: int = 10) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    curve_rows = []

    for task, payload in tasks.items():
        y_true = payload["y_true"]
        y_prob = np.clip(payload["y_prob"], 1e-6, 1 - 1e-6)
        observed = float(np.mean(y_true)) if len(y_true) else np.nan
        predicted = float(np.mean(y_prob)) if len(y_prob) else np.nan
        expected_observed_ratio = observed / predicted if predicted > 0 else np.nan

        summary_rows.append(
            {
                "task": task,
                "label": payload["label"],
                "n": int(len(y_true)),
                "events": int(np.sum(y_true)),
                "event_rate": observed,
                "mean_predicted_probability": predicted,
                "observed_expected_ratio": expected_observed_ratio,
                "brier_score": float(brier_score_loss(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan,
            }
        )

        bin_df = pd.DataFrame({"y_true": y_true, "y_prob": y_prob})
        unique_prob_count = bin_df["y_prob"].nunique()
        if len(bin_df) and unique_prob_count > 1:
            bin_count = min(n_bins, unique_prob_count)
            bin_df["bin"] = pd.qcut(bin_df["y_prob"], q=bin_count, duplicates="drop")
            grouped = bin_df.groupby("bin", observed=True)
            for bin_idx, (_, group) in enumerate(grouped, start=1):
                curve_rows.append(
                    {
                        "task": task,
                        "label": payload["label"],
                        "bin": bin_idx,
                        "n": int(len(group)),
                        "events": int(group["y_true"].sum()),
                        "mean_predicted_probability": float(group["y_prob"].mean()),
                        "observed_event_rate": float(group["y_true"].mean()),
                        "min_predicted_probability": float(group["y_prob"].min()),
                        "max_predicted_probability": float(group["y_prob"].max()),
                    }
                )

    return pd.DataFrame(summary_rows), pd.DataFrame(curve_rows)


def decision_curve(tasks: dict, thresholds: np.ndarray) -> pd.DataFrame:
    rows = []
    for task, payload in tasks.items():
        y_true = payload["y_true"].astype(int)
        y_prob = payload["y_prob"].astype(float)
        n = len(y_true)
        prevalence = float(np.mean(y_true)) if n else np.nan

        for threshold in thresholds:
            if threshold <= 0 or threshold >= 1 or n == 0:
                continue
            y_pred = y_prob >= threshold
            tp = int(np.sum(y_pred & (y_true == 1)))
            fp = int(np.sum(y_pred & (y_true == 0)))
            odds = threshold / (1.0 - threshold)
            model_net_benefit = (tp / n) - (fp / n) * odds
            treat_all_net_benefit = prevalence - (1.0 - prevalence) * odds
            rows.extend(
                [
                    {
                        "task": task,
                        "label": payload["label"],
                        "threshold": float(threshold),
                        "strategy": "model",
                        "net_benefit": float(model_net_benefit),
                        "standardized_net_benefit": float(model_net_benefit / prevalence) if prevalence > 0 else np.nan,
                    },
                    {
                        "task": task,
                        "label": payload["label"],
                        "threshold": float(threshold),
                        "strategy": "treat_all",
                        "net_benefit": float(treat_all_net_benefit),
                        "standardized_net_benefit": float(treat_all_net_benefit / prevalence) if prevalence > 0 else np.nan,
                    },
                    {
                        "task": task,
                        "label": payload["label"],
                        "threshold": float(threshold),
                        "strategy": "treat_none",
                        "net_benefit": 0.0,
                        "standardized_net_benefit": 0.0,
                    },
                ]
            )
    return pd.DataFrame(rows)


def continuous_nri(y_true: np.ndarray, new_prob: np.ndarray, ref_prob: np.ndarray) -> dict:
    y_true = y_true.astype(int)
    delta = new_prob - ref_prob
    event = y_true == 1
    nonevent = y_true == 0

    event_up = float(np.mean(delta[event] > 0)) if event.any() else np.nan
    event_down = float(np.mean(delta[event] < 0)) if event.any() else np.nan
    nonevent_down = float(np.mean(delta[nonevent] < 0)) if nonevent.any() else np.nan
    nonevent_up = float(np.mean(delta[nonevent] > 0)) if nonevent.any() else np.nan
    event_component = event_up - event_down if event.any() else np.nan
    nonevent_component = nonevent_down - nonevent_up if nonevent.any() else np.nan

    return {
        "continuous_event_up": event_up,
        "continuous_event_down": event_down,
        "continuous_nonevent_down": nonevent_down,
        "continuous_nonevent_up": nonevent_up,
        "continuous_event_component": event_component,
        "continuous_nonevent_component": nonevent_component,
        "continuous_nri": event_component + nonevent_component,
    }


def categorical_nri(y_true: np.ndarray, new_prob: np.ndarray, ref_prob: np.ndarray, cutpoints: list[float]) -> dict:
    y_true = y_true.astype(int)
    bins = [-np.inf, *cutpoints, np.inf]
    new_category = np.digitize(new_prob, bins[1:-1], right=False)
    ref_category = np.digitize(ref_prob, bins[1:-1], right=False)
    delta_category = new_category - ref_category
    event = y_true == 1
    nonevent = y_true == 0

    event_up = float(np.mean(delta_category[event] > 0)) if event.any() else np.nan
    event_down = float(np.mean(delta_category[event] < 0)) if event.any() else np.nan
    nonevent_down = float(np.mean(delta_category[nonevent] < 0)) if nonevent.any() else np.nan
    nonevent_up = float(np.mean(delta_category[nonevent] > 0)) if nonevent.any() else np.nan
    event_component = event_up - event_down if event.any() else np.nan
    nonevent_component = nonevent_down - nonevent_up if nonevent.any() else np.nan

    return {
        "risk_cutpoints": ",".join(map(str, cutpoints)),
        "categorical_event_up": event_up,
        "categorical_event_down": event_down,
        "categorical_nonevent_down": nonevent_down,
        "categorical_nonevent_up": nonevent_up,
        "categorical_event_component": event_component,
        "categorical_nonevent_component": nonevent_component,
        "categorical_nri": event_component + nonevent_component,
    }


def load_reference_probabilities(reference_path: Path, meta: pd.DataFrame) -> tuple[str, np.ndarray | None]:
    if reference_path is None or not reference_path.exists():
        return "train_prevalence", None

    reference_df = pd.read_csv(reference_path)
    if len(reference_df) != len(meta):
        raise ValueError(f"Reference prediction row count mismatch: {len(reference_df)} != {len(meta)}")
    if "example_id" in reference_df.columns and "example_id" in meta.columns:
        if not reference_df["example_id"].astype(str).equals(meta["example_id"].astype(str)):
            raise ValueError("Reference prediction example_id order does not match meta_test")

    reference_prob = np.zeros((len(reference_df), len(HORIZONS)), dtype=np.float32)
    for idx, horizon in enumerate(HORIZONS):
        column = f"{horizon}_prob"
        if column not in reference_df.columns:
            raise ValueError(f"Missing reference prediction column: {column}")
        reference_prob[:, idx] = reference_df[column].astype(float).to_numpy()
    return NRI_REFERENCE_MODEL_NAME, reference_prob


def build_prevalence_reference_tasks(y_steps: np.ndarray, y_mask: np.ndarray) -> dict:
    reference = {}
    for idx, horizon in enumerate(HORIZONS):
        active = y_mask[:, idx].astype(bool)
        reference[horizon] = float(np.mean(y_steps[active, idx])) if active.any() else np.nan
    reference["within_t_plus_2"] = float(np.mean(((y_steps * y_mask).max(axis=1) > 0).astype(int)))
    return reference


def reference_tasks_from_probability_matrix(reference_prob: np.ndarray | None) -> dict | None:
    if reference_prob is None:
        return None
    return build_prediction_tasks(y_test_steps, reference_prob, y_test_step_mask)


def net_reclassification_table(
    model_tasks: dict,
    reference_tasks: dict | None,
    prevalence_reference: dict,
    reference_name: str,
    cutpoints: list[float],
) -> pd.DataFrame:
    rows = []
    for task, payload in model_tasks.items():
        y_true = payload["y_true"]
        new_prob = payload["y_prob"]
        if reference_tasks is None:
            ref_prob = np.repeat(prevalence_reference[task], len(new_prob)).astype(float)
        else:
            ref_prob = reference_tasks[task]["y_prob"].astype(float)

        row = {
            "task": task,
            "label": payload["label"],
            "reference": reference_name,
            "n": int(len(y_true)),
            "events": int(np.sum(y_true)),
            "mean_model_probability": float(np.mean(new_prob)),
            "mean_reference_probability": float(np.mean(ref_prob)),
        }
        row.update(continuous_nri(y_true, new_prob, ref_prob))
        row.update(categorical_nri(y_true, new_prob, ref_prob, cutpoints))
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
test_tasks = build_prediction_tasks(y_test_steps, test_prob, y_test_step_mask)
calibration_summary, calibration_curve = calibration_table(test_tasks, n_bins=CALIBRATION_N_BINS)
dca_results = decision_curve(test_tasks, thresholds=DCA_THRESHOLDS)

reference_name, reference_prob = load_reference_probabilities(NRI_REFERENCE_PREDICTION_PATH, meta_test)
reference_tasks = reference_tasks_from_probability_matrix(reference_prob)
train_prevalence_reference = build_prevalence_reference_tasks(y_train_steps, y_train_step_mask)
nri_results = net_reclassification_table(
    test_tasks,
    reference_tasks=reference_tasks,
    prevalence_reference=train_prevalence_reference,
    reference_name=reference_name,
    cutpoints=NRI_RISK_CUTPOINTS,
)

calibration_summary.to_csv(OUTPUT_DIR / "xgb_multi_horizon_calibration_summary.csv", index=False)
calibration_curve.to_csv(OUTPUT_DIR / "xgb_multi_horizon_calibration_curve.csv", index=False)
dca_results.to_csv(OUTPUT_DIR / "xgb_multi_horizon_decision_curve.csv", index=False)
nri_results.to_csv(OUTPUT_DIR / "xgb_multi_horizon_nri.csv", index=False)

print("NRI reference:", reference_name)
display(calibration_summary)
display(nri_results)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for task, payload in test_tasks.items():
    curve = calibration_curve[calibration_curve["task"] == task]
    if curve.empty:
        continue
    axes[0].plot(
        curve["mean_predicted_probability"],
        curve["observed_event_rate"],
        marker="o",
        linewidth=1.8,
        label=payload["label"],
    )
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="perfect")
axes[0].set_title("Calibration")
axes[0].set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Observed event rate")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].legend(loc="best")

for task, payload in test_tasks.items():
    model_curve = dca_results[(dca_results["task"] == task) & (dca_results["strategy"] == "model")]
    if model_curve.empty:
        continue
    axes[1].plot(
        model_curve["threshold"],
        model_curve["net_benefit"],
        linewidth=1.8,
        label=payload["label"],
    )

within_all_curve = dca_results[
    (dca_results["task"] == "within_t_plus_2") & (dca_results["strategy"] == "treat_all")
]
within_none_curve = dca_results[
    (dca_results["task"] == "within_t_plus_2") & (dca_results["strategy"] == "treat_none")
]
axes[1].plot(within_all_curve["threshold"], within_all_curve["net_benefit"], linestyle="--", color="gray", linewidth=1, label="treat all")
axes[1].plot(within_none_curve["threshold"], within_none_curve["net_benefit"], linestyle=":", color="black", linewidth=1, label="treat none")
axes[1].set_title("Decision curve analysis")
axes[1].set_xlabel("Threshold probability")
axes[1].set_ylabel("Net benefit")
axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.5)
axes[1].legend(loc="best")

fig.tight_layout()
fig.savefig(FIGURE_DIR / "xgb_multi_horizon_calibration_dca.png", dpi=200, bbox_inches="tight")
plt.show()

nri_plot = nri_results.set_index("label")[["continuous_nri", "categorical_nri"]]
fig, ax = plt.subplots(figsize=(8, 4.5))
nri_plot.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title(f"Net reclassification improvement vs {reference_name}")
ax.set_ylabel("NRI")
ax.set_xlabel("")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "xgb_multi_horizon_nri.png", dpi=200, bbox_inches="tight")
plt.show()


## 해석 대상 subset 선택


In [ ]:
X_explain_source = X_test
y_explain_steps = y_test_steps
y_explain_mask = y_test_step_mask
meta_explain = meta_test

n_explain = min(EXPLAIN_SAMPLE_SIZE, len(X_explain_source))
explain_idx = rng.choice(len(X_explain_source), size=n_explain, replace=False)
X_explain = X_explain_source[explain_idx].copy()
y_explain_steps_sub = y_explain_steps[explain_idx].copy()
y_explain_mask_sub = y_explain_mask[explain_idx].copy()
meta_explain_sub = meta_explain.iloc[explain_idx].reset_index(drop=True)

baseline_prob = predict_proba(horizon_models, X_explain)
baseline_metrics = masked_metric_summary(y_explain_steps_sub, baseline_prob, y_explain_mask_sub)
print("explain subset:", X_explain.shape)
display(pd.DataFrame([baseline_metrics]))


## Permutation feature importance


In [ ]:
def permutation_feature_importance(
    horizon_models: dict,
    x: np.ndarray,
    y_steps: np.ndarray,
    y_mask: np.ndarray,
    feature_names: list[str],
    repeats: int,
) -> pd.DataFrame:
    baseline_prob = predict_proba(horizon_models, x)
    baseline = masked_metric_summary(y_steps, baseline_prob, y_mask)
    feature_indices = list(range(x.shape[1]))

    rows = []
    for feature_idx in feature_indices:
        repeat_rows = []
        for repeat in range(1, repeats + 1):
            x_perm = x.copy()
            perm = rng.permutation(x_perm.shape[0])
            x_perm[:, feature_idx] = x_perm[perm, feature_idx]
            perm_prob = predict_proba(horizon_models, x_perm)
            metrics = masked_metric_summary(y_steps, perm_prob, y_mask)
            repeat_rows.append(metrics)

        repeat_df = pd.DataFrame(repeat_rows)
        row = {
            "feature_idx": feature_idx,
            "feature": feature_names[feature_idx],
            "repeats": repeats,
        }
        for metric_name, baseline_value in baseline.items():
            if metric_name.endswith("_n"):
                continue
            perm_mean = float(repeat_df[metric_name].mean())
            perm_std = float(repeat_df[metric_name].std(ddof=0))
            row[f"baseline_{metric_name}"] = baseline_value
            row[f"permuted_mean_{metric_name}"] = perm_mean
            row[f"permuted_std_{metric_name}"] = perm_std
            row[f"drop_{metric_name}"] = baseline_value - perm_mean
        rows.append(row)

    return pd.DataFrame(rows)


feature_importance = permutation_feature_importance(
    horizon_models=horizon_models,
    x=X_explain,
    y_steps=y_explain_steps_sub,
    y_mask=y_explain_mask_sub,
    feature_names=feature_columns,
    repeats=PERMUTATION_REPEATS,
)
feature_importance = feature_importance.sort_values("drop_within_t_plus_2_auprc", ascending=False).reset_index(drop=True)
feature_importance.to_csv(OUTPUT_DIR / "xgb_multi_horizon_permutation_feature_importance.csv", index=False)
display(feature_importance.head(30))


In [ ]:
plot_df = feature_importance.head(TOP_N_PLOT).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(plot_df))))
ax.barh(plot_df["feature"], plot_df["drop_within_t_plus_2_auprc"], color="tab:blue")
ax.set_xlabel("AUPRC drop after permutation")
ax.set_ylabel("Feature")
ax.set_title("XGBoost multi-horizon permutation feature importance")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "xgb_multi_horizon_permutation_feature_importance_top.png", dpi=200, bbox_inches="tight")
plt.show()


## XGBoost built-in gain importance


In [ ]:
gain_rows = []
for horizon, estimator in horizon_models.items():
    booster = estimator.get_booster()
    score = booster.get_score(importance_type="gain")
    for key, value in score.items():
        if key.startswith("f") and key[1:].isdigit():
            feature_idx = int(key[1:])
            feature = feature_columns[feature_idx]
        else:
            feature_idx = np.nan
            feature = key
        gain_rows.append(
            {
                "horizon": horizon,
                "horizon_label": HORIZON_LABELS.get(horizon, horizon),
                "feature_idx": feature_idx,
                "feature": feature,
                "gain": float(value),
            }
        )

xgb_gain_importance = pd.DataFrame(gain_rows)
xgb_gain_importance["gain_rank_within_horizon"] = xgb_gain_importance.groupby("horizon")["gain"].rank(ascending=False, method="first").astype(int)
xgb_gain_importance = xgb_gain_importance.sort_values(["horizon", "gain_rank_within_horizon"])
xgb_gain_importance.to_csv(OUTPUT_DIR / "xgb_multi_horizon_gain_feature_importance.csv", index=False)
display(xgb_gain_importance.groupby("horizon").head(15))


In [ ]:
top_gain = xgb_gain_importance[xgb_gain_importance["gain_rank_within_horizon"] <= TOP_N_PLOT].copy()
for horizon in HORIZONS:
    plot_df = top_gain[top_gain["horizon"] == horizon].sort_values("gain").copy()
    fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(plot_df))))
    ax.barh(plot_df["feature"], plot_df["gain"], color="tab:green")
    ax.set_xlabel("Gain")
    ax.set_ylabel("Feature")
    ax.set_title(f"XGBoost gain importance ({HORIZON_LABELS.get(horizon, horizon)})")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"xgb_multi_horizon_gain_importance_{horizon}.png", dpi=200, bbox_inches="tight")
    plt.show()


## SHAP

`RUN_SHAP = True`로 설정한 경우 선택한 horizon의 XGBoost tree model을 Tree SHAP으로 설명합니다.


In [ ]:
print("RUN_SHAP:", RUN_SHAP)

if RUN_SHAP:
    import shap

    if SHAP_HORIZON not in horizon_models:
        raise ValueError(f"Unknown SHAP_HORIZON: {SHAP_HORIZON}")

    ex_n = min(SHAP_EXPLAIN_SIZE, len(X_explain))
    shap_x = X_explain[:ex_n]
    shap_model = horizon_models[SHAP_HORIZON]
    explainer = shap.TreeExplainer(shap_model)
    shap_values = explainer.shap_values(shap_x)
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    mean_abs_by_feature = np.abs(shap_values).mean(axis=0)
    shap_feature_importance = pd.DataFrame(
        {
            "feature_idx": np.arange(len(feature_columns)),
            "feature": feature_columns,
            "mean_abs_shap": mean_abs_by_feature,
            "horizon": SHAP_HORIZON,
            "horizon_label": HORIZON_LABELS.get(SHAP_HORIZON, SHAP_HORIZON),
        }
    ).sort_values("mean_abs_shap", ascending=False)
    shap_feature_importance.to_csv(OUTPUT_DIR / "xgb_multi_horizon_shap_feature_importance.csv", index=False)
    display(shap_feature_importance.head(30))

    plot_df = shap_feature_importance.head(TOP_N_PLOT).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(plot_df))))
    ax.barh(plot_df["feature"], plot_df["mean_abs_shap"], color="tab:purple")
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Feature")
    ax.set_title(f"XGBoost SHAP feature importance ({HORIZON_LABELS.get(SHAP_HORIZON, SHAP_HORIZON)})")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "xgb_multi_horizon_shap_feature_importance_top.png", dpi=200, bbox_inches="tight")
    plt.show()


## 저장된 산출물


In [ ]:
saved_files = sorted([str(path.relative_to(PROJECT_DIR)) for path in OUTPUT_DIR.glob("xgb_multi_horizon*.csv")])
saved_figures = sorted([str(path.relative_to(PROJECT_DIR)) for path in FIGURE_DIR.glob("xgb_multi_horizon*.png")])
print("XGBoost CSV outputs:")
for path in saved_files:
    print("-", path)
print("XGBoost figure outputs:")
for path in saved_figures:
    print("-", path)
